## Базовый пример запуска Gamac

### Импортируем нужные библиотеки

In [1]:
import sys
sys.path.append('../')

import warnings
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.datasets import load_digits
from torchvision.datasets import CIFAR100

from gamac.autoclustering import Gamac
from gamac.estimation.internal import Internal

### Получим данные для кластеризации

In [4]:
# Импортируем датафрейм для табличных данных
data = load_digits(as_frame=True)

table = data['data']
table.head()
targets = data['target'].values

In [ ]:
# Импортируем данные для CIFAR
# Первая загрузка будет долгая так как он грузит из open-source
cifar100 = CIFAR100('../data/cifar', download=True, train=False)

cifar_txt = [f'a photo of {cifar100.classes[img[1]]}' for img in cifar100]
cifar_img = [img[0] for img in cifar100]
cifar_table = pd.DataFrame(cifar100.targets)

100%|██████████| 169M/169M [00:10<00:00, 15.8MB/s] 


### Инициализируем Gamac
- Класс Gamac имеет следующие аргументы:
    - mab_solver: MabSolvers = MabSolvers.SOFTMAX - тип алгоритма multi-arm bandit
    - hyper_optimiser: HyperOptimisers = HyperOptimisers.OPTUNA - Тип оптимизатора, по умолчанию Optuna
    - target_measures: list[str] | None = None - Целевая мера, по умолчанию не указывается - в этом случае происходит автоматический подбор меры
    - time_limit: int | None = None - Время в секундах поиска оптимальной модели кластеризации
    - iter_limit: int | None = 50 - Кол-во итераций поиска оптимальной модели кластеризации
    - batch_size: int = 32 - размер батча для предобработки текста и изображений
    - model_name: str = "openai/clip-vit-large-patch14" - CLIP-based модель для обработки изображений и текста

In [ ]:
# В качестве примера ограничивается количество итераций поиска до 30
gamac = Gamac(iter_limit=30)
# На реальных наборах данных рекомендуется использовать ограничение по времени (time_limit)
# Для больших наборов данных и/или для лучшего качества кластеризации нужно увеличивать time_limit
# Например, ограничение временного бюджета на 30 минут
# gamac = Gamac(time_limit=1800)

### Запустим поиск оптимального алгоритма кластеризации для табличных данных
Работа происходит в три этапа
1. Обработка и приведение модальностей в единый вектор
2. Определение меры - можно задать явно в target_measures, либо оставить пустым (в этом случае мера будет подобрана автоматически)
3. Поиск алгоритма кластеризации с наилучшими по выбранным мерам гиперпараметрами
- В ходе перебора логируются успешные и неуспешные запуски перебора
- Неуспешные запуски появляются в ходе получения некорректных гиперпараметров алгоритма под конкретный датасет

### Базовый пример для табличных данных

#### При запуске Gamac.run выдаются логи выполнения работы
- """CVI prediction iteration %N""" - этот лог выводит номер итерации %N подбора итоговой меры
- """Picked %measure in %time""" - этот лог отображает итоговое время %time для подбора финальной меры %measure из возможных:
    - BR: Метрика Banfield-Raftery
    - OS: Метрика относительной разделимости
    - MCR: Метрика McClain-Rao
    - SYM: Симметричная метрика
- """MEASURES %time, {%measure: %score}""" - этот лог оценки по выбранной мере %measure с его значением %score и временем оценки %time по алгоритму выводимому логом ниже
- """ALGO %time, %status, {%configuration}""" - этот лог выбранной модели, где
    - %time - это время запуска в секундах
    - %status - статус обучения и использования модели с выбранными гиперпараметрами
    - %configuration - выбранная модель с гиперпараметрами

In [7]:
result = gamac.run(table=table, classes=targets)

=== Started CVI prediction ===
=== CVI prediction iteration 1/8 ====
=== CVI prediction iteration 2/8 ====
=== CVI prediction iteration 3/8 ====
=== CVI prediction iteration 4/8 ====
=== CVI prediction iteration 5/8 ====
=== CVI prediction iteration 6/8 ====
=== CVI prediction iteration 7/8 ====
=== CVI prediction iteration 8/8 ====
=== Picked SYM in 13.691501140594482s ===
=== MEASURES 1.3876936435699463s, {'SYM': 7.0957054343766905e-06} ===
=== MEASURES 0.01330423355102539s, {'SYM': 0.010814519187061276} ===
=== ALGO 7.783405065536499s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 15, 'max_iter': 100, 'init': 'random', 'tol': 0.0001} ===
=== ALGO 3.849851131439209s, FailedRun, {'bandwidth': 0.6891579613584354, 'max_iter': 216, 'tol': 7.379942820597754e-05} ===
=== ALGO 0.8843080997467041s, FailedRun, {'name': 'DBSCAN', 'eps': 0.2660228290876068, 'eps_sq': 0.07076814559577407, 'min_samples': 9} ===
=== ALGO 4.411066293716431s, FailedRun, {'min_cluster_size': 13, 'min_samples'

### На выходе получаем итоговый датасет после обработки

In [8]:
print(result.df.shape)
print(result.df[:10])

(1797, 64)
[[ 0.0000000e+00 -3.3501649e-01 -4.3081019e-02  2.7407151e-01
  -6.6447753e-01 -8.4412938e-01 -4.0972391e-01 -1.2502292e-01
  -5.9077557e-02 -6.2400925e-01  4.8297450e-01  7.5962245e-01
  -5.8425862e-02  1.1277211e+00  8.7958306e-01 -1.3043338e-01
  -4.4625074e-02  1.1144272e-01  8.9588046e-01 -8.6066633e-01
  -1.1496484e+00  5.1547188e-01  1.9059634e+00 -1.1422184e-01
  -3.3379726e-02  4.8648927e-01  4.6988511e-01 -1.4999014e+00
  -1.6140628e+00  7.6397777e-02  1.5418141e+00 -4.7232382e-02
   0.0000000e+00  7.6465553e-01  5.2630186e-02 -1.4476300e+00
  -1.7366644e+00  4.3615878e-02  1.4395580e+00  0.0000000e+00
  -6.1343670e-02  8.1055361e-01  6.3011712e-01 -1.1224571e+00
  -1.0662316e+00  6.6096473e-01  8.1845075e-01 -8.8741615e-02
  -3.5433263e-02  7.4211895e-01  1.1506522e+00 -8.6867058e-01
   1.1012974e-01  5.3761119e-01 -7.5743580e-01 -2.0978513e-01
  -2.3596458e-02 -2.9908136e-01  8.6718693e-02  2.0829257e-01
  -3.6677122e-01 -1.1466475e+00 -5.0566983e-01 -1.9600752e-

### Также получаем информацию о лучшей модели - ее наилучшей конфигурацией и финального значения мер

In [9]:
print(result.estimation) # Значение внутренней меры
print(result.f1_score) # Значение внешней меры при наличии разметки

{<Internal.SYM: ('sym', <function sym at 0x7f887eb9d940>)>: 0.010814519187061276}
0.6549112970142008


### Получение меток по лучшему классификатору

In [10]:
print(result.labels)

[ 6 13 14 ... 14  1 11]


### Пример для мультимодальных данных
Для этого используем данные по CIFAR100

### В данном примере можно использовать все модальности (таблица, текст и изображение)

In [11]:
# В данном примере можно использовать все модальности (таблица, текст и изображение)
gamac = Gamac(iter_limit=30, target_measures={Internal.SYM})
result = gamac.run(table=cifar_table, text=cifar_txt, image=cifar_img)

print(result.labels)

=== MEASURES 0.004900693893432617s, {'SYM': 0.0055735762366099645} ===
=== MEASURES 0.0045430660247802734s, {'SYM': 0.04743209204922543} ===
=== ALGO 0.4002683162689209s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 2, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 1.6435601711273193s, FailedRun, {'bandwidth': 0.9177592749762231, 'max_iter': 288, 'tol': 9.202271134864323e-05} ===
=== ALGO 0.029757261276245117s, FailedRun, {'name': 'DBSCAN', 'eps': 0.4054546256750648, 'eps_sq': 0.1643934534813069, 'min_samples': 10} ===
=== MEASURES 0.004949331283569336s, {'SYM': 0.018973797371520797} ===
=== ALGO 0.4712200164794922s, SuccessRun, {'min_cluster_size': 6, 'min_samples': 12} ===
=== MEASURES 0.005192756652832031s, {'SYM': 0.14305628004814255} ===
=== ALGO 0.5556449890136719s, SuccessRun, {'threshold': 0.8958850732475786, 'branching_factor': 67, 'n_clusters': 4} ===
=== ALGO 0.014381647109985352s, FailedRun, {'name': 'KMeans', 'n_clusters': 8, 'max_iter': 100, 't

### или использовать только текст + изображение

In [12]:
result = gamac.run(text=cifar_txt, image=cifar_img)

print(result.labels)

=== MEASURES 0.00757598876953125s, {'SYM': 0.004481222199823049} ===
=== MEASURES 0.006959438323974609s, {'SYM': 0.09409667341767658} ===
=== ALGO 0.05263686180114746s, SuccessRun, {'name': 'BisectingKMeans', 'n_clusters': 5, 'max_iter': 100, 'init': 'k-means++', 'tol': 0.0001} ===
=== ALGO 0.0964357852935791s, FailedRun, {'bandwidth': 0.18962519247437493, 'max_iter': 205, 'tol': 2.7155745649999224e-05} ===
=== ALGO 0.5558285713195801s, FailedRun, {'name': 'DBSCAN', 'eps': 0.7785186159263283, 'eps_sq': 0.6060912353438459, 'min_samples': 8} ===
=== ALGO 0.3294227123260498s, FailedRun, {'min_cluster_size': 14, 'min_samples': 5} ===
=== ALGO 0.25781750679016113s, FailedRun, {'threshold': 0.13765581390632883, 'branching_factor': 22, 'n_clusters': 15} ===
=== ALGO 0.019399404525756836s, FailedRun, {'name': 'KMeans', 'n_clusters': 15, 'max_iter': 100, 'tol': 0.0001, 'random_state': None} ===
=== ALGO 0.33821630477905273s, FailedRun, {'name': 'BisectingKMeans', 'n_clusters': 15, 'max_iter': 1

### Таким образом Gamac может подбирать кластеризацию для:
1. табличных
2. изображений с текстом
3. совместно